# ORE XVA Master Results

## Aggregation & PostProcess Bindings Demonstration

This notebook demonstrates the full Python bindings for the ORE aggregation and
post-processing layer. It runs an XVA workflow on a multi-currency swap portfolio
and programmatically extracts:

- **Trade-level exposure profiles** (EPE, ENE, PFE, EE_B, EEE_B)
- **Netting-set exposure profiles** (net EPE/ENE/PFE, expected collateral)
- **XVA scalars** (CVA, DVA, FBA, FCA, MVA, COLVA, CollateralFloor)
- **Allocated exposures** (EPE/ENE/CVA/DVA by trade)

### Prerequisites
- Python 3.10+
- ORE Python module: `pip install open-source-risk-engine`

## 1. Setup & Run ORE

In [ ]:
import sys, os
sys.path.insert(0, '..')
from ORE import OREApp, Parameters

# Load parameters and run XVA workflow
params = Parameters()
params.fromFile('Input/ore.xml')
app = OREApp(params)
app.run()

errors = app.getErrors()
print(f'Run time: {app.getRunTime():.2f} sec')
print(f'Errors:   {len(errors)}')
for e in errors:
    print(f'  ERROR: {e}')

## 2. Access PostProcess via XVA Analytic

The `PostProcess` object is the central result store after an XVA simulation.
We access it through the XVA analytic's `postProcess()` accessor.

In [ ]:
# Get PostProcess from the XVA analytic
analytic = app.getAnalytic('XVA')
pp = analytic.postProcess()

# Summary
trade_ids = sorted(dict(pp.tradeIds()).keys())
ns_ids = sorted(dict(pp.nettingSetIds()).keys())
cpty_map = dict(pp.counterpartyId())

print(f'Trade IDs:       {trade_ids}')
print(f'Netting Sets:    {ns_ids}')
print(f'Counterparties:  {cpty_map}')
print(f'Portfolio size:  {pp.portfolio().size()}')

## 3. Trade-Level Exposure Profiles

Extract EPE, ENE, PFE, and Basel EE profiles per trade.

In [ ]:
import pandas as pd

for tid in trade_ids:
    epe = list(pp.tradeEPE(tid))
    ene = list(pp.tradeENE(tid))
    pfe = list(pp.tradePFE(tid))
    ee_b = list(pp.tradeEE_B(tid))
    eee_b = list(pp.tradeEEE_B(tid))
    
    print(f'\n--- Trade: {tid} ---')
    print(f'  EPE_B (Basel):  {pp.tradeEPE_B(tid):.6f}')
    print(f'  EEPE_B (Basel): {pp.tradeEEPE_B(tid):.6f}')
    
    df = pd.DataFrame({'EPE': epe, 'ENE': ene, 'PFE': pfe, 'EE_B': ee_b, 'EEE_B': eee_b})
    display(df.head(10))

## 4. Netting-Set Exposure Profiles

Net EPE, ENE, PFE and expected collateral at the netting-set level.

In [ ]:
for ns in ns_ids:
    epe = list(pp.netEPE(ns))
    ene = list(pp.netENE(ns))
    pfe = list(pp.netPFE(ns))
    coll = list(pp.expectedCollateral(ns))
    
    print(f'\n--- Netting Set: {ns} ---')
    print(f'  Net EPE_B:  {pp.netEPE_B(ns):.6f}')
    print(f'  Net EEPE_B: {pp.netEEPE_B(ns):.6f}')
    
    df = pd.DataFrame({'Net_EPE': epe, 'Net_ENE': ene, 'Net_PFE': pfe, 'Expected_Collateral': coll})
    display(df.head(10))

## 5. Trade-Level XVA Scalars

CVA, DVA, FBA, FCA, and MVA per trade.

In [ ]:
xva_data = []
for tid in trade_ids:
    xva_data.append({
        'Trade': tid,
        'CVA': pp.tradeCVA(tid),
        'DVA': pp.tradeDVA(tid),
        'FBA': pp.tradeFBA(tid),
        'FCA': pp.tradeFCA(tid),
        'MVA': pp.tradeMVA(tid),
    })

df_xva = pd.DataFrame(xva_data).set_index('Trade')
display(df_xva)

## 6. Netting-Set XVA Scalars

Full XVA breakdown at the netting-set level including COLVA and CollateralFloor.

In [ ]:
for ns in ns_ids:
    print(f'\n--- Netting Set: {ns} ---')
    print(f'  CVA:              {pp.nettingSetCVA(ns):>14.2f}')
    print(f'  DVA:              {pp.nettingSetDVA(ns):>14.2f}')
    print(f'  FBA:              {pp.nettingSetFBA(ns):>14.2f}')
    print(f'  FCA:              {pp.nettingSetFCA(ns):>14.2f}')
    print(f'  MVA:              {pp.nettingSetMVA(ns):>14.2f}')
    print(f'  COLVA:            {pp.nettingSetCOLVA(ns):>14.2f}')
    print(f'  CollateralFloor:  {pp.nettingSetCollateralFloor(ns):>14.2f}')
    print(f'  FBA (ex own SP):  {pp.nettingSetFBA_exOwnSP(ns):>14.2f}')
    print(f'  FCA (ex own SP):  {pp.nettingSetFCA_exOwnSP(ns):>14.2f}')
    print(f'  FBA (ex all SP):  {pp.nettingSetFBA_exAllSP(ns):>14.2f}')
    print(f'  FCA (ex all SP):  {pp.nettingSetFCA_exAllSP(ns):>14.2f}')

## 7. Allocated Exposures & XVA

Marginal allocation of netting-set exposure down to individual trades.

In [ ]:
alloc_data = []
for tid in trade_ids:
    alloc_epe = list(pp.allocatedTradeEPE(tid))
    alloc_ene = list(pp.allocatedTradeENE(tid))
    alloc_data.append({
        'Trade': tid,
        'Alloc_CVA': pp.allocatedTradeCVA(tid),
        'Alloc_DVA': pp.allocatedTradeDVA(tid),
        'Alloc_EPE_peak': max(alloc_epe) if alloc_epe else 0,
        'Alloc_ENE_peak': min(alloc_ene) if alloc_ene else 0,
    })

df_alloc = pd.DataFrame(alloc_data).set_index('Trade')
display(df_alloc)

## 9. Summary

This notebook demonstrated programmatic access to all PostProcess
result APIs from Python, covering:

| Category | Methods Used |
|----------|-------------|
| Trade Exposure | `tradeEPE`, `tradeENE`, `tradePFE`, `tradeEE_B`, `tradeEEE_B`, `tradeEPE_B`, `tradeEEPE_B` |
| Net Exposure | `netEPE`, `netENE`, `netPFE`, `expectedCollateral`, `netEPE_B`, `netEEPE_B` |
| Trade XVA | `tradeCVA`, `tradeDVA`, `tradeFBA`, `tradeFCA`, `tradeMVA` |
| Net XVA | `nettingSetCVA/DVA/FBA/FCA/MVA/COLVA/CollateralFloor` |
| Allocation | `allocatedTradeEPE/ENE`, `allocatedTradeCVA/DVA` |
| Metadata | `tradeIds`, `nettingSetIds`, `counterpartyId`, `portfolio` |